# Description-Logic Reasoners (HermiT/Pellet)

**Domain:** Symbolic AI & Logic  ·  **recommended addition**  ·  **runnable:** no — conceptual / CLI / snippets  ·  _Java reasoners, driven via OWL API / Protégé / ROBOT_

> A self-contained refresher on **OWL description-logic (DL) reasoners** — the engines that *compute the logical consequences* of an OWL ontology: **HermiT** (hypertableau), **Pellet / Openllet** (tableau, with explanations + SWRL + ABox query), plus **FaCT++**, **ELK**, and **Konclude**. They classify your class hierarchy, prove it's consistent, find unsatisfiable classes, and infer individuals' types — all under the **Open World Assumption**. This is the *inference* sequel to [`owl.ipynb`](owl.ipynb) (the language) and sits beside [`knowledge-graphs.ipynb`](knowledge-graphs.ipynb) / [`sparql.ipynb`](sparql.ipynb) (data & query) and [`answer-set-programming.ipynb`](answer-set-programming.ipynb) / [`datalog.ipynb`](datalog.ipynb) (the other logic substrates).
>
> _Not Python-runnable from a notebook cell: these reasoners are **Java** programs you drive through the OWL API, the Protégé GUI, the ROBOT CLI, or the `owlready2` Python bridge (which shells out to a bundled HermiT/Pellet and needs a JRE). All code below is **real CLI / config / snippets** — no `print()` theater._

## 1. What & Why

**What it is.** A DL reasoner is a **theorem prover for a decidable fragment of first-order logic** — the description logic **`SROIQ(D)`** that underpins **OWL 2 DL**. You hand it an ontology (axioms + facts); it computes what *logically follows*. The standard reasoning services:

- **Consistency** — is the whole ontology satisfiable (does *some* model exist)? An *inconsistent* ontology entails everything and is useless.
- **Classification** — compute the complete **subsumption** hierarchy: for every pair of named classes, does `C ⊑ D` (C is necessarily a subclass of D)? This is the inferred `is-a` tree Protégé shows in yellow.
- **Class satisfiability** — can a class have any instance, or is it **unsatisfiable** (equivalent to `owl:Nothing`)? Unsatisfiable classes are almost always modeling bugs.
- **Realization / instance checking** — for each individual, compute its **most specific inferred types**; answer "is `a` a `C`?".
- **Entailment & query** — does the ontology entail an axiom? Some reasoners (Pellet/Openllet) answer **conjunctive ABox queries** (SPARQL-DL) under entailment, not just over asserted triples.

**The problem it solves.** OWL lets you *state* rich logical relationships (disjointness, cardinality, property chains, restrictions), but the *consequences* of those statements are not written down — they have to be **derived**. Doing that by hand is infeasible and error-prone; the reasoner does it **soundly, completely, and terminating-ly**. It turns a pile of axioms into (a) a verdict that your model is coherent and (b) a fully inferred hierarchy + typing you can query, validate against, or export.

**Reach for it when** you have an OWL ontology and need the *inferred* class hierarchy, want to catch contradictions/unsatisfiable classes early, need entailment-based typing of individuals, or want machine-checkable "this design is logically coherent." Classic homes: biomedical ontologies (SNOMED CT, GO), configuration/product modeling, data integration schemas.

**Skip it when** you only need to *query asserted data* (use a triple store + [`sparql.ipynb`](sparql.ipynb)), need *scalable forward-chaining materialization* over billions of triples (use an OWL **RL** rule engine like RDFox/GraphDB, not a DL tableau), or want *closed-world constraint validation* (use **SHACL** — see Gotchas). DL reasoning is worst-case **N2ExpTime-complete**; expressivity has a price.

## 2. Mental Model

**A DL reasoner is a model-builder that proves things by failing to find a counterexample.** To test whether `C ⊑ D`, it doesn't search forward for a proof — it tries to **construct a model** (a "tableau": a graph of individuals satisfying all axioms) in which something is a `C` but **not** a `D`, i.e. it tries to satisfy `C ⊓ ¬D`. If every attempt hits a logical **clash** (e.g. an individual forced to be both `X` and `¬X`), no counter-model exists, so the subsumption **holds**. Consistency is the same move on the whole ontology; satisfiability of `C` is "can I build any individual that is a `C`?".

```
      TBox  (terminology / schema)          ABox  (assertions / data)
   ┌──────────────────────────────┐   ┌──────────────────────────────┐
   │ Pizza ⊑ Food                  │   │ MyDinner : VegetarianPizza    │
   │ VegetarianPizza ≡ Pizza ⊓      │   │ MyDinner hasTopping  m1       │
   │   ∀hasTopping.(¬MeatTopping)  │   │ m1 : MozzarellaTopping        │
   │ MeatTopping ⊓ VegTopping ⊑ ⊥  │   └──────────────────────────────┘
   └──────────────┬───────────────┘                  │
                  └───────────────┬──────────────────┘
                                  ▼
                 ┌──────────────────────────────────┐
                 │   DL REASONER  (tableau / hyper-  │
                 │   tableau / consequence-based)    │
                 │   try to build a model; clash =   │
                 │   entailment holds                │
                 └──────────────┬───────────────────┘
                                ▼
      consistent? ──── classified hierarchy ──── inferred types ──── explanations
      (yes/no)         (C ⊑ D for all C,D)        (MyDinner is Food)   (why C is ⊥)
```

Two assumptions that govern *everything* the reasoner concludes:

- **Open World Assumption (OWA):** what you didn't state is **unknown**, not false. "We have no record that Pizza X has a meat topping" ≠ "Pizza X has no meat topping." You must **close** the world explicitly (e.g. `owl:allValuesFrom`, cardinality, `owl:oneOf`) to get closed-world-style conclusions.
- **No Unique Name Assumption (no UNA):** two differently-named individuals **may be the same** unless you assert `owl:differentFrom` (or a cardinality/`owl:AllDifferent` forces it). This routinely surprises people writing cardinality restrictions.

**Slogan:** *the TBox says what's possible, the reasoner finds what's necessary — by trying, and failing, to imagine otherwise.*

## 3. Key Concepts

| Term | What it means |
| --- | --- |
| **TBox / RBox / ABox** | **TBox** = class axioms (terminology: `⊑`, `≡`, disjointness, restrictions). **RBox** = property axioms (domain/range, sub-properties, `owl:TransitiveProperty`, property chains, inverses). **ABox** = assertions about individuals (`a : C`, `a R b`). |
| **`SROIQ(D)`** | The description logic OWL 2 DL maps to. Letters = features: **S** (transitive roles), **R** (role chains/hierarchies), **O** (nominals/`oneOf`), **I** (inverse roles), **Q** (qualified cardinality), **(D)** (datatypes). Each feature adds cost. |
| **Subsumption / classification** | `C ⊑ D`: every instance of C is necessarily a D. **Classification** = computing this for all named classes → the inferred hierarchy. |
| **Satisfiability** | A class is **satisfiable** if it *can* have an instance. **Unsatisfiable** (`≡ owl:Nothing`) means contradictory definition — a bug. Distinct from ontology **(in)consistency**. |
| **Consistency** | The ontology has at least one model. An **inconsistent** ontology entails *everything* (`⊥`-explosion) — usually triggered when an individual is asserted into an unsatisfiable class. |
| **Realization** | Computing each individual's **most specific** inferred types (its position in the classified hierarchy). |
| **Entailment** | `O ⊨ α`: the ontology logically implies axiom α. All services reduce to entailment / (un)satisfiability. |
| **OWA / no UNA** | Open World + No Unique Names (see §2) — the two assumptions behind "surprising" inferences. |
| **OWL 2 profiles** | Sub-languages traded for tractability: **EL** (PTIME, big bio-ontologies → use **ELK**), **QL** (query rewriting over DBs), **RL** (rule/materialization, OWA-lite). Full **DL** needs HermiT/Pellet/FaCT++/Konclude. |
| **Calculus** | How the reasoner decides: **tableau** (Pellet, FaCT++), **hypertableau** (HermiT — fewer non-deterministic branches), **consequence-based** (ELK — EL only, very fast). |
| **Justification / explanation** | A **minimal set of axioms** responsible for an entailment (e.g. *why* a class is unsatisfiable). Pellet/Openllet + the OWL API compute these; Protégé shows them. |
| **Incremental reasoning** | Re-classify after small edits without full recompute (Pellet supports it) — matters in interactive editors. |
| **OWL API `OWLReasoner`** | The Java interface every reasoner implements (`isConsistent`, `getSubClasses`, `getTypes`, `isEntailed`, …). Swapping HermiT↔Pellet = swapping a factory. |

## 4. Setup

**Not Python-runnable from a notebook.** These are **Java** reasoners (`SROIQ` tableau engines). You drive them four ways: (1) the **Protégé** desktop GUI, (2) the **OWL API** in Java, (3) the **ROBOT** command-line tool, or (4) the **`owlready2`** Python library, which bundles HermiT + Pellet as jars and *shells out to a JRE*. All four need **Java 8+ on the `PATH`** — there is no pure-Python DL tableau reasoner to import and run in this kernel, which is exactly why this notebook is `runnable: false`.

```bash
# 0. Prereq: a JRE/JDK (every option below calls Java under the hood)
java -version          # need 8+, 11+ recommended

# --- Option A: Protégé (GUI) — easiest way to *see* reasoning ---
#   Download from https://protege.stanford.edu ; bundles HermiT.
#   Reasoner menu -> Start reasoner ; inferred (yellow) axioms appear in the class tree.

# --- Option B: ROBOT (CLI, great for pipelines/CI) ---
#   Apache-licensed, wraps the OWL API + ELK/HermiT. https://robot.obolibrary.org
curl -L https://github.com/ontodev/robot/releases/latest/download/robot.jar -o robot.jar
echo 'java -jar '"$PWD"'/robot.jar "$@"' > robot && chmod +x robot   # tiny wrapper
./robot --version

# --- Option C: owlready2 (Python bridge; STILL needs Java for the reasoner) ---
pip install owlready2          # bundles HermiT + Pellet jars
#   owlready2.sync_reasoner()  -> writes a temp file, runs HermiT, reads inferences back.

# --- Option D: raw reasoner jars for the OWL API (Java) ---
#   HermiT:   http://www.hermit-reasoner.com  (org.semanticweb.HermiT)
#   Openllet: https://github.com/Galigator/openllet  (maintained Pellet fork)
#   FaCT++ / ELK / Konclude: see §8
```

**Which reasoner?** Default to **HermiT** for full OWL 2 DL (robust, hypertableau). Use **Pellet/Openllet** when you need **explanations**, **SWRL** rules, **incremental** reasoning, or **SPARQL-DL** ABox queries. Use **ELK** when the ontology is **OWL 2 EL** and huge (SNOMED CT, GO) — it's orders of magnitude faster but only supports the EL profile. **Konclude** is the speed champion for expressive DL in benchmarks.

**Authoring is data-first.** You spend your time editing the ontology (`.owl`/`.ttl`/`.ofn`), then *running a reasoner* to see what it entails — not writing imperative code. The snippets in §5 reflect that loop.

## 5. Worked Examples

Conceptual walkthroughs — **not executed in this kernel** (the reasoners are Java; this notebook is `runnable: false`). The ontology snippets are valid OWL; the CLI/Java/Python is faithful to each tool's real API.

### Example 1 — Classify a tiny ontology (ROBOT + HermiT)

Goal: state only the *definitions* and let the reasoner **derive** that a margherita is a `VegetarianPizza` and a `Food`. Ontology in Turtle (`pizza.ttl`):

```turtle
@prefix :    <http://ex.org/pizza#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs:<http://www.w3.org/2000/01/rdf-schema#> .

:Pizza        a owl:Class ; rdfs:subClassOf :Food .
:Topping      a owl:Class .
:MeatTopping  a owl:Class ; rdfs:subClassOf :Topping .
:VegTopping   a owl:Class ; rdfs:subClassOf :Topping .
:MeatTopping  owl:disjointWith :VegTopping .

# A VegetarianPizza is DEFINED as a pizza all of whose toppings are non-meat:
:VegetarianPizza a owl:Class ;
  owl:equivalentClass [ a owl:Class ;
    owl:intersectionOf ( :Pizza
      [ a owl:Restriction ; owl:onProperty :hasTopping ;
        owl:allValuesFrom [ owl:complementOf :MeatTopping ] ] ) ] .

:hasTopping a owl:ObjectProperty .
:Mozzarella a owl:Class ; rdfs:subClassOf :VegTopping .

# Assert a pizza and CLOSE its toppings (OWA! see Gotchas):
:Margherita a :Pizza ;
  :hasTopping [ a :Mozzarella ] ;
  rdfs:subClassOf [ a owl:Restriction ; owl:onProperty :hasTopping ;
                    owl:allValuesFrom :Mozzarella ] .
```

Run classification and write an ontology with the inferences materialized:

```bash
# Reason with HermiT, then assert the inferred subclass/type axioms into the output
./robot reason   --input pizza.ttl --reasoner hermit \
        --output pizza-reasoned.ttl
./robot reason   --input pizza.ttl --reasoner hermit \
        --axiom-generators "subclass equivalent classassertion" \
        --output pizza-inferred.ttl
```

**Inferred (not asserted):** `Margherita ⊑ VegetarianPizza` (its toppings are closed to non-meat `Mozzarella`), and transitively `Margherita ⊑ Food`. Note the `owl:allValuesFrom :Mozzarella` **closure axiom** — without it, OWA means the reasoner can't rule out a hidden meat topping, and the vegetarian inference **does not** fire.

### Example 2 — Catch a bug: an unsatisfiable class + its justification (Pellet/Protégé)

Goal: show how the reasoner finds modeling errors. Add a contradictory class:

```turtle
# A "MeatyVeggiePizza" that is required to have a topping that is BOTH meat and veg:
:MeatyVeggiePizza a owl:Class ;
  rdfs:subClassOf :Pizza ,
    [ a owl:Restriction ; owl:onProperty :hasTopping ; owl:someValuesFrom :MeatTopping ] ,
    [ a owl:Restriction ; owl:onProperty :hasTopping ; owl:allValuesFrom :VegTopping ] .
```

Because `MeatTopping` and `VegTopping` are **disjoint**, no topping can satisfy both `someValuesFrom MeatTopping` and `allValuesFrom VegTopping` → the class is **unsatisfiable** (`≡ owl:Nothing`). The reasoner flags it:

```bash
# ROBOT fails the build if any class is unsatisfiable — perfect for CI
./robot reason --input pizza.ttl --reasoner hermit
# ERROR  Unsatisfiable classes: [pizza#MeatyVeggiePizza]   (non-zero exit code)
```

To get the **justification** (the minimal axiom set to blame), use Pellet/Openllet via the OWL API (Protégé surfaces the same thing in its "Explain" pane):

```java
// Java + OWL API + Openllet (Pellet): explain WHY a class is unsatisfiable
OWLReasonerFactory rf = new openllet.owlapi.OpenlletReasonerFactory();
OWLReasoner r = rf.createReasoner(ontology);

boolean ok = r.isConsistent();                      // ontology-level check
boolean sat = r.isSatisfiable(meatyVeggiePizza);    // false here
Set<OWLClass> bad = r.getUnsatisfiableClasses()     // bottom node = all unsat classes
                     .getEntitiesMinusBottom();

// Minimal justification via the OWL API explanation API:
ExplanationGenerator<OWLAxiom> gen =
    new InconsistentOntologyExplanationGeneratorFactory(rf, () -> df).createExplanationGenerator(ontology);
// -> {MeatTopping disjointWith VegTopping, MeatyVeggiePizza ⊑ ∃hasTopping.MeatTopping,
//     MeatyVeggiePizza ⊑ ∀hasTopping.VegTopping}   // exactly the 3 axioms at fault
```

**Lesson:** an *unsatisfiable class* is latent; it becomes a full **inconsistent ontology** (entails everything) the moment you assert `:x a :MeatyVeggiePizza`. Classify early and often.

### Example 3 — Drive HermiT from Python via `owlready2` (snippet — needs a JRE, not run here)

`owlready2` is the usual Python on-ramp: it builds the ontology in memory, **writes a temp file, runs the HermiT jar in a subprocess**, and reads the inferences back. It is *not* a Python reasoner — `sync_reasoner()` raises if `java` isn't found, which is why this cell is shown, not executed.

```python
# NOT EXECUTED in this notebook (requires Java). Conceptually:
from owlready2 import (get_ontology, Thing, ObjectProperty,
                       AllDisjoint, Not, sync_reasoner_hermit)

onto = get_ontology("http://ex.org/pizza#")
with onto:
    class Food(Thing): pass
    class Pizza(Food): pass
    class Topping(Thing): pass
    class MeatTopping(Topping): pass
    class VegTopping(Topping): pass
    AllDisjoint([MeatTopping, VegTopping])
    class hasTopping(ObjectProperty): pass
    class Mozzarella(VegTopping): pass

    class VegetarianPizza(Pizza):
        equivalent_to = [Pizza & hasTopping.only(Not(MeatTopping))]

    m = Pizza("Margherita")
    m.hasTopping = [Mozzarella()]
    m.is_a.append(hasTopping.only(Mozzarella))   # close the world for this individual

sync_reasoner_hermit(infer_property_values=True)  # shells out to HermiT
print(VegetarianPizza in m.is_a)                   # -> True (INFERRED, not asserted)
print(Food in m.INDIRECT_is_a)                     # -> True (transitive)
# If you'd asserted m into an unsatisfiable class, this raises OwlReadyInconsistentOntologyError.
```

### Example 4 — Swap reasoners behind one interface (OWL API, Java)

The whole point of the `OWLReasoner` interface: the *service calls* are identical; only the **factory** changes. Pick the engine to fit the profile/feature you need.

```java
OWLOntologyManager m = OWLManager.createOWLOntologyManager();
OWLOntology o = m.loadOntologyFromOntologyDocument(new File("pizza.owl"));

// Choose ONE factory — the rest of the code is reasoner-agnostic:
OWLReasonerFactory rf = new org.semanticweb.HermiT.ReasonerFactory();        // full DL
// OWLReasonerFactory rf = new openllet.owlapi.OpenlletReasonerFactory();    // DL + SWRL + explain
// OWLReasonerFactory rf = new org.semanticweb.elk.owlapi.ElkReasonerFactory(); // EL only, fast

OWLReasoner r = rf.createReasoner(o);
r.precomputeInferences(InferenceType.CLASS_HIERARCHY);   // classify

System.out.println(r.isConsistent());                    // consistency check
NodeSet<OWLClass> subs = r.getSubClasses(vegetarianPizza, /*direct=*/false);  // classification
NodeSet<OWLClass> types = r.getTypes(margherita,          /*direct=*/true);    // realization
boolean entailed = r.isEntailed(df.getOWLSubClassOfAxiom(margherita_cls, food)); // entailment
```

Use ELK's factory and the same code runs in PTIME on an EL ontology — but throws if the ontology uses a feature outside EL (e.g. inverse roles, disjunction). That trade — **expressivity vs speed** — is the whole reasoner-selection game.

## 6. Gotchas & Pitfalls

- **Open World bites the most.** Newcomers expect "I didn't say Pizza X has a meat topping" to mean "it has none." It doesn't — the reasoner keeps that possibility open. You only get closed-world conclusions by **closing** the world: `owl:allValuesFrom`, exact/max cardinality, `owl:oneOf`, or explicit negation. Example 1's vegetarian inference *only* fires because of the closure axiom.
- **No Unique Name Assumption + cardinality = surprises.** `p hasParent a, b` with `hasParent max 1` does **not** make the ontology inconsistent — the reasoner concludes `a = b` (they were never assumed distinct). Assert `owl:differentFrom` / `owl:AllDifferent` when you mean it.
- **Unsatisfiable class ≠ inconsistent ontology.** A contradictory *class* is fine until you put an *individual* in it; then the **whole ontology** is inconsistent and entails everything (every class subsumes every other, every individual has every type). Always classify and inspect the "bottom node" before trusting any inference.
- **`rdfs:domain` / `rdfs:range` are inferences, not constraints.** They don't *reject* bad data — they **add types**. `hasTopping rdfs:domain Pizza` + `Car hasTopping x` makes the reasoner infer `Car ⊑ Pizza` (and possibly an inconsistency), not raise a validation error. For *constraint* checking use **SHACL** (closed-world, validation), not DL domain/range.
- **Expressivity is exponential.** `SROIQ` is N2ExpTime-complete; nominals (`O`), inverse roles (`I`), and qualified cardinality (`Q`) interacting can make HermiT crawl or blow memory. If your ontology fits **OWL 2 EL**, switch to **ELK** for a massive speedup; profile-check with `./robot validate-profile --profile EL`.
- **Not every reasoner supports every feature.** **SWRL rules** and **SPARQL-DL/ABox query** → Pellet/Openllet, not HermiT. **Datatype** reasoning support varies. **ELK** silently ignores or rejects axioms outside EL — verify your ontology is in-profile or you'll get incomplete results.
- **Datatypes & nominals are easy to get wrong.** Custom datatype restrictions and `owl:oneOf` enumerations are supported but expensive and a frequent source of "why is this unsatisfiable?" confusion. Add them last; classify after each.
- **`owlready2`/ROBOT/Protégé all need Java.** "It works in my notebook" fails on a box without a JRE; `sync_reasoner()` raises a cryptic error. Pin the Java version; reasoning is also **non-incremental by default** (full re-classify per run) unless you use Pellet's incremental mode.
- **Reasoning ≠ querying asserted data.** A DL reasoner answers *entailment* questions; a plain SPARQL endpoint answers over *asserted* triples. For "give me all inferred instances of C," either **materialize** the inferences first (`robot reason` → load into a store) or use a reasoner with **SPARQL-DL** — don't expect a vanilla triple store to do OWL DL entailment.

## 7. When to Use vs Alternatives

| Approach | Good at | Trade-off |
| --- | --- | --- |
| **HermiT** (hypertableau) | Robust **full OWL 2 DL** classification/consistency; fewer non-deterministic branches than classic tableau; the Protégé default | No SWRL, no ABox/SPARQL-DL query; expressive ontologies can still be slow |
| **Pellet / Openllet** (tableau) | **Explanations/justifications**, **SWRL** rules, **incremental** reasoning, **SPARQL-DL** ABox query; Openllet is the maintained fork | Heavier; original Pellet unmaintained — use **Openllet** |
| **FaCT++** (tableau, C++) | Fast classification of expressive TBoxes; long pedigree | Native lib (JNI) — packaging/portability friction; weaker ABox/explanations |
| **ELK** (consequence-based) | **OWL 2 EL** at huge scale (SNOMED CT, GO) — PTIME, parallel, very fast | EL only — drops axioms using disjunction, inverses, cardinality, negation |
| **Konclude** (saturation+tableau, C++) | Benchmark-leading speed on expressive DL | Standalone/native; less of a library-embedding story than the OWL API engines |
| **OWL 2 RL engine** (RDFox, GraphDB, Jena rules) | **Materialize** entailments over **billions** of triples via forward-chaining rules | Incomplete for full DL (RL profile only); rule semantics, not tableau — different guarantees |
| **OWL 2 QL + query rewriting** (Ontop) | Ontology-based access over **relational DBs**; scales with the DB | QL profile only; answers queries, doesn't classify rich TBoxes |
| **SHACL** ([`knowledge-graphs.ipynb`](knowledge-graphs.ipynb)) | **Closed-world validation** ("every Person must have exactly one SSN") with clear violation reports | Validation, not inference/classification; opposite world assumption to OWL |
| **Datalog / ASP** ([`datalog.ipynb`](datalog.ipynb), [`answer-set-programming.ipynb`](answer-set-programming.ipynb)) | Closed-world rules, recursion, defaults/negation, combinatorial search | Not OWL-DL semantics; no built-in subsumption/OWA; you encode the logic yourself |
| **SPARQL** ([`sparql.ipynb`](sparql.ipynb)) | Querying asserted graphs; property paths give *some* transitive reach | No DL entailment — won't classify or detect unsatisfiability |

**Rules of thumb.** Need the **inferred class hierarchy / consistency / unsatisfiable-class detection** on an expressive ontology → **HermiT** (add **Pellet/Openllet** when you need *why*, SWRL, or ABox queries). Ontology is **EL** and large → **ELK**. Need entailments over **huge data**, not a rich schema → an **OWL RL** materializer (RDFox/GraphDB). Want to **enforce** shape constraints with closed-world reports → **SHACL**. Just **querying** what's asserted → **SPARQL**. The deciding axis is almost always **expressivity (OWL profile) × scale × which question** (classify vs validate vs query).

## 8. Resources

**Reasoners**
- **HermiT** — homepage + downloads (Oxford, hypertableau): http://www.hermit-reasoner.com/
- **Openllet** — maintained Pellet fork (explanations, SWRL, SPARQL-DL): https://github.com/Galigator/openllet
- **ELK** — OWL 2 EL reasoner (SNOMED-scale): https://github.com/liveontologies/elk-reasoner
- **FaCT++**: https://github.com/ethz-asl/libfactplusplus  ·  **Konclude**: https://www.derivo.de/en/products/konclude/

**Tools to drive them**
- **Protégé** — ontology editor with built-in reasoning + "Explain" justifications: https://protege.stanford.edu/
- **ROBOT** — command-line OWL toolkit (`reason`, `validate-profile`) for pipelines/CI: https://robot.obolibrary.org/
- **owlready2** — Python ontology library + reasoner bridge (docs): https://owlready2.readthedocs.io/
- **OWL API** — the Java library + `OWLReasoner` interface every reasoner implements: https://github.com/owlcs/owlapi

**Specs & background**
- **OWL 2 Primer** (W3C) — gentle intro to the language the reasoners decide: https://www.w3.org/TR/owl2-primer/
- **OWL 2 Profiles** (EL/QL/RL and why they're tractable): https://www.w3.org/TR/owl2-profiles/
- **OWL 2 Direct Semantics** (what "entails" formally means): https://www.w3.org/TR/owl2-direct-semantics/
- **The hypertableau paper** — *HermiT: An OWL 2 Reasoner* (Glimm, Horrocks, Motik et al., JAR 2014): https://link.springer.com/article/10.1007/s10817-014-9305-1
- *The Description Logic Handbook* (Baader et al.) — the canonical reference for the underlying logic.

**Cross-links in this library:** [`owl.ipynb`](owl.ipynb) (the OWL language these reason over) · [`knowledge-graphs.ipynb`](knowledge-graphs.ipynb) / [`rdflib.ipynb`](rdflib.ipynb) (RDF data + SHACL) · [`sparql.ipynb`](sparql.ipynb) (querying, asserted vs entailed) · [`datalog.ipynb`](datalog.ipynb) / [`answer-set-programming.ipynb`](answer-set-programming.ipynb) (closed-world logic alternatives).